In [1]:
%pip install selenium 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import openpyxl

In [3]:
import os
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys

In [4]:
Datefile = pd.read_excel(r'C:\Users\KS\Desktop\Python\DATE.xlsx')
Datefile['DATE'] = pd.to_datetime(Datefile['DATE']).dt.date

In [5]:
kmart = 'KMART3'

In [6]:
download_dir = fr"C:\Users\KS\Desktop\{kmart}"
# สร้างโฟลเดอร์ถ้ายังไม่มี
if not os.path.exists(download_dir):
    os.makedirs(download_dir)
# --- ฟังก์ชันจัดการเปลี่ยนชื่อไฟล์ ---
def wait_and_rename(new_name):
    timeout = 45 # เพิ่มเวลารอเป็น 45 วินาที (เผื่อเน็ตช้า)
    start_time = time.time()
    
    print(f"กำลังค้นหาไฟล์เพื่อเปลี่ยนชื่อเป็น: {new_name}...")
    
    while time.time() - start_time < timeout:
        # 1. รายชื่อไฟล์ทั้งหมดในโฟลเดอร์
        files = os.listdir(download_dir)
        
        # 2. กรองเฉพาะไฟล์จริงที่โหลดเสร็จแล้ว (ต้องไม่มี .crdownload หรือ .tmp)
        # และต้องเป็นไฟล์ที่ขึ้นต้นด้วย 'DiffList' ตามที่คุณส่งรูปมา
        actual_files = [f for f in files if f.startswith("DiffList") and not f.endswith(('.crdownload', '.tmp'))]
        
        if actual_files:
            # เจอไฟล์เป้าหมายแล้ว!
            old_file_name = actual_files[0]
            old_path = os.path.join(download_dir, old_file_name)
            
            # ตรวจสอบนามสกุล (คงเดิมไว้)
            extension = os.path.splitext(old_file_name)[1]
            new_file_name = f"{new_name}{extension}"
            new_path = os.path.join(download_dir, new_file_name)
            
            # เปลี่ยนชื่อ (ถ้ามีชื่อซ้ำให้ลบก่อน)
            try:
                if os.path.exists(new_path):
                    os.remove(new_path)
                os.rename(old_path, new_path)
                return True
            except Exception as e:
                # บางครั้งไฟล์ยังโดน Chrome ล็อคไว้ (กำลังย้ายจาก Temp มา Folder จริง)
                print(f"ไฟล์ยังไม่พร้อมเปลี่ยนชื่อ (กำลังลองใหม่...): {e}")
        
        time.sleep(1.5) # พักเช็คทุก 1.5 วินาที
    return False

In [7]:

# 1. ตั้งค่า Options (ใส่เพื่อให้รันได้เสถียรขึ้น)
chrome_options = Options()

# ตั้งค่าให้ Chrome ให้ดาวโหลดไฟล์เลยไม่ต้องถาม
prefs = {
    "download.default_directory": download_dir , # กำหนดที่เก็บไฟล์ (ใส่ path เต็ม)
    "download.prompt_for_download": False,                # ไม่ต้องถามว่าจะเซฟที่ไหน
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True                           # ต้องเปิดไว้นิดนึงเพื่อตั้งค่าตัวถัดไป
}

chrome_options.add_experimental_option("prefs", prefs)

# 2. ปิดตัวกรองไฟล์อันตรายเฉพาะหน้า (สำคัญมากสำหรับกรณีนี้)
chrome_options.add_argument("--safebrowsing-disable-download-protection")
chrome_options.add_argument("--allow-running-insecure-content")
chrome_options.add_argument("--ignore-certificate-errors")
chrome_options.add_argument("--unsafely-treat-insecure-origin-as-secure=http://49.0.94.69")
# 2. เริ่มต้น WebDriver (Selenium จะหา Chrome และ Driver ให้เองอัตโนมัติ)
driver = webdriver.Chrome(options=chrome_options)
wait = WebDriverWait(driver, 10)
try:
    # 3. สั่งให้ไปที่เว็บไซต์ที่ต้องการ
    url = "http://49.0.94.69"
    driver.get(url)
    time.sleep(3)
    chick =  wait.until(EC.element_to_be_clickable((By.XPATH , '/html/body/div[1]/div/div[2]/nav/ul/li[3]/a')))
    chick.click()
    user =  wait.until(EC.element_to_be_clickable((By.XPATH , '/html/body/div[2]/div/form/fieldset/div[1]/div/input')))
    user.send_keys('kit')
    passd =  wait.until(EC.element_to_be_clickable((By.XPATH , '/html/body/div[2]/div/form/fieldset/div[2]/div/input')))
    passd.send_keys('290745')
    passd.send_keys(Keys.ENTER)
    # สั่งให้รอสูงสุด 10 วินาที จนกว่าปุ่ม 'รับสินค้า' (ตาม XPATH) จะปรากฏและคลิกได้
    #ปุ่มรายการสินค้า
    รับสินค้า = wait.until(EC.element_to_be_clickable((By.XPATH, "//a[contains(text(), 'รับสินค้า')]")))
    รับสินค้า.click()
    #ปุ่มDiff  
    Diff = wait.until(EC.element_to_be_clickable((By.XPATH, "//a[contains(text(), 'DIFF')]")))
    Diff.click()
    #ปุ่มจำนวน
    จำนวน = wait.until(EC.element_to_be_clickable((By.XPATH, "//a[contains(text(), 'จำนวน')]")))
    จำนวน.click()
    #ปุ่มเลือกตำแหน่ง Kmart
    Kmart = wait.until(EC.element_to_be_clickable((By.XPATH,"/html/body/div[2]/form[1]/table/tbody/tr/td[1]/select/option[3]")))
    Kmart.click()    
    #ปุ่ม ALL               
    all =  wait.until(EC.element_to_be_clickable((By.XPATH, "/html/body/div[2]/form[1]/table/tbody/tr/td[2]/select/option[3]")))
    all.click()

    Diff2 = wait.until(EC.element_to_be_clickable((By.XPATH, "/html/body/div[2]/form[1]/table/tbody/tr/td[3]/select/option[3]")))
    Diff2.click()

    time.sleep(3)
    
    for i in Datefile['DATE']:
        iso_date = i.strftime('%Y-%m-%d')
        file_name = i.strftime('%d-%m-%Y') # ใช้ - แทน / สำหรับตั้งชื่อไฟล์
    
        date_el = wait.until(EC.presence_of_element_located((By.NAME, "DateExport")))
        driver.execute_script("arguments[0].value = arguments[1];", date_el, iso_date)
        driver.execute_script("arguments[0].dispatchEvent(new Event('change', { bubbles: true }));", date_el)

        print(f"\n--- เริ่มจัดการวันที่: {file_name} ---")
        time.sleep(1)
    
        # 5. กดปุ่มตกลง
        ok_btn = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'ตกลง')]")))
        ok_btn.click()

        # 6. Export 
        try:
            # รอจนกว่าแถวข้อมูลจะปรากฏ (รอสูงสุด 10 วิ)
            wait.until(EC.presence_of_element_located((By.XPATH, "/html/body/div[2]/form[2]/table/tbody/tr[1]")))
            print("พบข้อมูลในตารางแล้ว...")
            time.sleep(1.5) # ให้เวลาตาราง Render ข้อมูลให้ครบจริงๆ

            # กดปุ่ม Export
            ex_btn = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Export')]")))
            ex_btn.click()
            print("กดปุ่ม Export แล้ว กำลังรอไฟล์ดาวน์โหลด...")

            # 7. รอโหลดและเปลี่ยนชื่อไฟล์ (ใช้ชื่อ file_name ที่ไม่มีเครื่องหมาย /)
            if wait_and_rename(file_name):
                print(f"✅ บันทึกไฟล์ {file_name}.xlsx เรียบร้อย")
            else:
                print(f"❌ ไฟล์ {file_name} ดาวน์โหลดล้มเหลว (Timeout)")

        except Exception as e:
            print(f"⚠️ วันที่ {file_name} ข้ามไป (อาจไม่มีข้อมูล หรือโหลดช้ามาก)")

        # 8. จุดสำคัญ: รอให้หน้าเว็บ "พร้อม" สำหรับรอบถัดไป
        # สั่งให้ Scroll กลับขึ้นไปข้างบนสุดเพื่อให้มั่นใจว่าปุ่ม 'ตกลง' ไม่อยู่ห่างสายตา Selenium
        driver.execute_script("window.scrollTo(0, 0);")
        wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'ตกลง')]")))
        time.sleep(1) # พักหายใจ 1 วินาทีก่อนวนลูปใหม่
        
except Exception as e:
    # เพิ่มบรรทัดนี้เพื่อดูว่ามันพังเพราะอะไรกันแน่!
    print(f"โปรแกรมหยุดทำงานเพราะ: {e}")


--- เริ่มจัดการวันที่: 01-12-2025 ---
พบข้อมูลในตารางแล้ว...
กดปุ่ม Export แล้ว กำลังรอไฟล์ดาวน์โหลด...
กำลังค้นหาไฟล์เพื่อเปลี่ยนชื่อเป็น: 01-12-2025...
✅ บันทึกไฟล์ 01-12-2025.xlsx เรียบร้อย

--- เริ่มจัดการวันที่: 02-12-2025 ---
พบข้อมูลในตารางแล้ว...
กดปุ่ม Export แล้ว กำลังรอไฟล์ดาวน์โหลด...
กำลังค้นหาไฟล์เพื่อเปลี่ยนชื่อเป็น: 02-12-2025...
✅ บันทึกไฟล์ 02-12-2025.xlsx เรียบร้อย

--- เริ่มจัดการวันที่: 03-12-2025 ---
พบข้อมูลในตารางแล้ว...
กดปุ่ม Export แล้ว กำลังรอไฟล์ดาวน์โหลด...
กำลังค้นหาไฟล์เพื่อเปลี่ยนชื่อเป็น: 03-12-2025...
✅ บันทึกไฟล์ 03-12-2025.xlsx เรียบร้อย

--- เริ่มจัดการวันที่: 04-12-2025 ---
พบข้อมูลในตารางแล้ว...
กดปุ่ม Export แล้ว กำลังรอไฟล์ดาวน์โหลด...
กำลังค้นหาไฟล์เพื่อเปลี่ยนชื่อเป็น: 04-12-2025...
✅ บันทึกไฟล์ 04-12-2025.xlsx เรียบร้อย

--- เริ่มจัดการวันที่: 05-12-2025 ---
พบข้อมูลในตารางแล้ว...
กดปุ่ม Export แล้ว กำลังรอไฟล์ดาวน์โหลด...
กำลังค้นหาไฟล์เพื่อเปลี่ยนชื่อเป็น: 05-12-2025...
✅ บันทึกไฟล์ 05-12-2025.xlsx เรียบร้อย

--- เริ่มจัดการวันที่: 06-12-

In [ ]:
driver.quit()